# Version 1: Simple RAG:-

Algorithm Sketch: Document Preparation:-
--------------------------------------------------
1. Load document and extract raw text
2. Split text into chunks of ~1000 characters with ~200 character overlap
3. Clean each chunk (remove formatting artifacts)
4. Convert each chunk into an embedding vector
5. Store all vectors in a vector store for fast similarit search

In [1]:
PDF_PATH = 'book/01_Designing_Machine_Learning_Systems.pdf'
JSON_PDF_STRUCTURE = 'book/book_structure.json'

EMBEDDING_MODEL = 'all-MiniLM-L6-v2'

CHROMA_DB_DIR = './chroma_db'

LLM_MODEL = 'llama3.2:3b'

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path=PDF_PATH)
documents = loader.load()

/home/mohamedhussam/miniconda3/envs/agentic-ai/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,      # characters per chunk
    chunk_overlap=200,    # overlap to preserve context
    separators=["\n\n", "\n", ".", " "]
)

chunks = splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")

Total chunks: 1145


In [ ]:
import pandas as pd
import numpy as np

structure_df = pd.read_json(JSON_PDF_STRUCTURE)

structure_df = structure_df.sort_values(by="page_number")


# ==========================================
# CLEAN FUNCTION
# ==========================================

def deep_clean(obj):

    if isinstance(obj, dict):
        return {k: deep_clean(v) for k, v in obj.items()}

    if isinstance(obj, (list, tuple)):
        return [deep_clean(x) for x in obj]

    if isinstance(obj, np.generic):
        obj = obj.item()

    if pd.isna(obj):
        return None

    return obj


# ==========================================
# STRUCTURE LOOKUP
# ==========================================

def get_structure_metadata(page_number):

    filtered = structure_df[
        structure_df["page_number"] <= page_number
    ]

    if filtered.empty:
        return {}

    latest = filtered.iloc[-1]

    return deep_clean({
        "chapter_number": latest["chapter_number"],
        "chapter_title": latest["chapter_title"],
        "section_title": latest["section_title"],
        "subsection_title": latest["subsection_title"],
        "hierarchy_path": latest["hierarchy_path"],
        "level": latest["level"],
    })


# ==========================================
# APPLY TO CHUNKS
# ==========================================

for chunk in chunks:

    # 1. CLEAN chunk first
    chunk.metadata = deep_clean(chunk.metadata)

    # 2. SAFE page number
    page_number = int(chunk.metadata.get("page_label", 0))

    # 3. GET structure metadata
    structure_metadata = get_structure_metadata(page_number)

    # 4. MERGE (structure overwrites if needed)
    chunk.metadata.update(structure_metadata)

In [4]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)  # Free & local

/tmp/ipykernel_347114/3064642783.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)  # Free & local
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8318.03it/s]


In [ ]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DB_DIR  # saves to disk
)

Algorithm Sketch: Query Processing:-
--------------------------------------------
1. Receive user question
2. Convert question into an embedding vector
3. Search vector store for top-K nearest chunk vectors
4. Retrieve the corresponding text chunks
5. Pass question + retrieved chunks to the language model
6. Model generates answer grounded in the retrieved context

In [ ]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from langchain_chroma import Chroma

def format_docs(docs):

    return "\n\n---\n\n".join(
        [
            f"[{d.metadata.get('hierarchy_path')}]\n{d.page_content}"
            for d in docs
        ]
    )
format_docs_runnable = RunnableLambda(format_docs)

llm = ChatOllama(
    model=LLM_MODEL,
    temperature=0
)


vectorstore = Chroma(persist_directory=CHROMA_DB_DIR, embedding_function=embeddings)

retriever = vectorstore.as_retriever(
    search_kwargs={"k": 5}
)


def build_prompt(x):

    return f"""
You are an expert assistant.

Answer ONLY using the context below.

Context:
{x['context']}

Question:
{x['question']}

Answer:
"""


prompt_runnable = RunnableLambda(build_prompt)

rag_chain = (
    {
        "context": retriever | format_docs_runnable,
        "question": RunnablePassthrough()
    }
    | prompt_runnable
    | llm
    | StrOutputParser()
)

In [6]:
import pandas as pd

import json

data = []
with open('jsonl/qa.jsonl') as f:
    for line in f:
        data.append(json.loads(line))
   

In [11]:
for q in data:
    print(q['question'])
    print()

    response = rag_chain.invoke(q['question'])
    print(response)

    print("#" * 50)
    print()

What is the difference between ML in research vs. production?

According to Table 1-1, there are five major differences between Machine Learning (ML) in Research and in Production:

Unfortunately, the table is not provided in the context, so I'll give a general answer based on the text.

In summary, ML in research focuses on understanding the underlying concepts and techniques of machine learning, often using complex models with large amounts of data. In contrast, ML in production aims to deploy models that can improve business outcomes, such as increasing sales or revenue. The focus shifts from accuracy metrics (e.g., 94% vs. 94.2%) to business metrics that drive real-world impact.
##################################################

What are the four main causes of data distribution shift?

The text doesn't explicitly list the four main causes of data distribution shift. However, it does mention some common causes:

1. Changes in social norms, cultures, languages, trends, industries, 